# 02 · Evaluación LightRAG
Evalúa LightRAG (Graph RAG + Gemini embeddings + Gemini LLM) sobre UltraDomain con métricas RAGAS.

In [2]:
import os
import sys
import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()
sys.path.append(os.path.abspath('../..'))
load_dotenv(os.path.join('../..', '.env'))

True

## 1. Configuración del experimento

In [7]:
DOMINIO       = "cs"
N_LIBROS      = None
MAX_Q         = None     # None = todas las preguntas
SHUFFLE       = False
LIGHTRAG_MODE = "hybrid" # "local" | "global" | "hybrid" | "naive"
WORKSPACE_DIR = "../../lightrag_eval"
RESULTS_DIR   = "../../results_LightRAG/"

## 2. Cargar datos

In [8]:
from src.ultradomain import cargar_experimento

libros, qas = cargar_experimento(DOMINIO, n_libros=N_LIBROS, shuffle=SHUFFLE)

📚 Dominio: cs
   📖 Guide to Java — James T. Streib (8 preguntas)
   📖 Professional Microsoft SQL Server 2008 Programming — Robert Vieira (11 preguntas)
   📖 Introducing Regular Expressions — Michael Fitzgerald (12 preguntas)
   📖 Linux Kernel Networking — Rami Rosen (8 preguntas)
   📖 Modern Optimization With R — Paulo Cortez (9 preguntas)
   📖 Probability and Statistics for Computer Science — David Forsyth (9 preguntas)
   📖 Joe Celko's SQL Programming Style — Joe Celko (11 preguntas)
   📖 Machine Learning With Spark — Nick Pentreath (8 preguntas)
   📖 Introduction to the Theory of Programming Languages — Gilles Dowek (9 preguntas)
   📖 Mastering VBA for Microsoft Office 2013 — Richard Mansfield (15 preguntas)
   Total Q&A: 100


## 3. Inicializar e indexar LightRAG

In [5]:
from src.baselines.lightrag_rag import build_lightrag, index_documents

rag, tracker = await build_lightrag(
    workspace_dir=WORKSPACE_DIR,
    clean=False,   # ← no limpiar, continuar desde donde se quedó
    max_retries=10,
)

INFO: [] Loaded graph from ../../lightrag_eval/graph_chunk_entity_relation.graphml with 54773 nodes, 72142 edges
INFO:nano-vectordb:Load (54773, 3072) data
INFO:nano-vectordb:Init {'embedding_dim': 3072, 'metric': 'cosine', 'storage_file': '../../lightrag_eval/vdb_entities.json'} 54773 data
INFO:nano-vectordb:Load (72142, 3072) data
INFO:nano-vectordb:Init {'embedding_dim': 3072, 'metric': 'cosine', 'storage_file': '../../lightrag_eval/vdb_relationships.json'} 72142 data
INFO:nano-vectordb:Load (3712, 3072) data
INFO:nano-vectordb:Init {'embedding_dim': 3072, 'metric': 'cosine', 'storage_file': '../../lightrag_eval/vdb_chunks.json'} 3712 data
INFO: [] Process 88107 KV load full_docs with 10 records
INFO: [] Process 88107 KV load text_chunks with 3712 records
INFO: [] Process 88107 KV load full_entities with 10 records
INFO: [] Process 88107 KV load full_relations with 10 records
INFO: [] Process 88107 KV load entity_chunks with 54773 records
INFO: [] Process 88107 KV load relation_chun

✅ LightRAG inicializado en: ../../lightrag_eval
   max_async=12 | max_retries=10 | modelo=gemini-2.5-flash-lite
   chunk_size=600 | gleaning=0


In [ ]:
await index_documents(rag, tracker, libros)

## 4. Ejecutar evaluación

In [9]:
from src.evaluation.experiment import run_experiment

result = await run_experiment(
    rag_type="lightrag",
    rag_object=rag,
    dominio=DOMINIO,
    n_libros=N_LIBROS,
    max_questions=MAX_Q,
    shuffle=SHUFFLE,
    lightrag_mode=LIGHTRAG_MODE,
    save_path=RESULTS_DIR,
)

📚 Dominio: cs
   📖 Guide to Java — James T. Streib (8 preguntas)
   📖 Professional Microsoft SQL Server 2008 Programming — Robert Vieira (11 preguntas)
   📖 Introducing Regular Expressions — Michael Fitzgerald (12 preguntas)
   📖 Linux Kernel Networking — Rami Rosen (8 preguntas)
   📖 Modern Optimization With R — Paulo Cortez (9 preguntas)
   📖 Probability and Statistics for Computer Science — David Forsyth (9 preguntas)
   📖 Joe Celko's SQL Programming Style — Joe Celko (11 preguntas)
   📖 Machine Learning With Spark — Nick Pentreath (8 preguntas)
   📖 Introduction to the Theory of Programming Languages — Gilles Dowek (9 preguntas)
   📖 Mastering VBA for Microsoft Office 2013 — Richard Mansfield (15 preguntas)
   Total Q&A: 100

🔍 Evaluando LIGHTRAG | cs | 100 preguntas
────────────────────────────────────────────────────────────
  [1/100] How does Spark Streaming enable real-time data processing?...


INFO: Query nodes: Spark Streaming, Apache Spark, D-Streams, RDDs, Micro-batching, Fault tolerance (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 408 relations
INFO: Query edges: Real-time data processing, Data streaming, Distributed computing (top_k:40, cosine:0.2)
INFO: Global query: 52 entites, 40 relations
INFO: Raw search results: 83 entities, 429 relations, 0 vector chunks
INFO: After truncation: 66 entities, 218 relations
INFO: Selecting 165 from 411 entity-related chunks by vector similarity
INFO: Find 3 additional chunks in 3 relations (deduplicated 82)
INFO: Selecting 3 from 3 relation-related chunks by vector similarity
INFO: Round-robin merged chunks: 168 -> 168 (deduplicated 0)
INFO: Final context: 66 entities, 218 relations, 20 chunks
INFO: Final chunks S+F/O: E11/1 R1/1 E6/2 R1/2 E7/3 R1/3 E3/4 E7/5 E4/6 E6/7 E4/8 E1/9 E4/10 E2/11 E10/12 E2/13 E4/14 E2/15 E4/16 E5/17
INFO:  == LLM cache == Query cache hit, using cached response as query result


  [2/100] What does the book suggest about the use of histograms in data analysi...


INFO: Query nodes: Book, Visualization, Data interpretation, Frequency distribution (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 131 relations
INFO: Query edges: Data analysis, Histograms, Statistical methods (top_k:40, cosine:0.2)
INFO: Global query: 51 entites, 40 relations
INFO: Raw search results: 84 entities, 148 relations, 0 vector chunks
INFO: After truncation: 62 entities, 148 relations
INFO: Selecting 155 from 463 entity-related chunks by vector similarity
INFO: Find 23 additional chunks in 22 relations (deduplicated 54)
INFO: Selecting 23 from 23 relation-related chunks by vector similarity
INFO: Round-robin merged chunks: 178 -> 178 (deduplicated 0)
INFO: Final context: 62 entities, 148 relations, 20 chunks
INFO: Final chunks S+F/O: E4/1 R1/1 E4/2 R1/2 E1/3 R3/3 E3/4 R1/4 E4/5 R1/5 E1/6 R1/6 E2/7 R1/7 E1/8 R1/8 E1/9 R2/9 E2/10 R1/10
INFO:  == LLM cache == Query cache hit, using cached response as query result


  [3/100] What are some advanced topics covered in the book related to Linux Ker...


...

  [100/100] How does the book introduce the concept of alternation in regular expr...


INFO: Query nodes: Alternation, Syntax, Pattern matching, Metacharacters (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 161 relations
INFO: Query edges: Regular expressions, Concept introduction (top_k:40, cosine:0.2)
INFO: Global query: 45 entites, 40 relations
INFO: Raw search results: 78 entities, 194 relations, 0 vector chunks
INFO: After truncation: 69 entities, 194 relations
INFO: Selecting 172 from 205 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 194 relations
INFO: Round-robin merged chunks: 172 -> 172 (deduplicated 0)
INFO: Final context: 69 entities, 194 relations, 20 chunks
INFO: Final chunks S+F/O: E6/1 E1/2 E2/3 E3/4 E18/5 E8/6 E4/7 E5/8 E10/9 E5/10 E22/11 E1/12 E2/13 E3/14 E9/15 E6/16 E1/17 E2/18 E1/19 E12/20
INFO:  == LLM cache == Query cache hit, using cached response as query result



📊 Calculando métricas RAGAS...
  RAGAS [1/100] How does Spark Streaming enable real-time data pro...
  RAGAS [2/100] What does the book suggest about the use of histog...
  RAGAS [3/100] What are some advanced topics covered in the book ...
  RAGAS [4/100] What is the significance of the R tool in the cont...
  RAGAS [5/100] What are the key features of this text that aid in...
  RAGAS [6/100] What is the role of the RegExr tool in the book?...
  RAGAS [7/100] How does the text compare to other Java programmin...
  RAGAS [8/100] What role do Bayesian inference and priors play in...
  RAGAS [9/100] What is the difference between recording a macro a...
  RAGAS [10/100] How does the book address the implementation of IP...
  RAGAS [11/100] Can you explain the concept of standard coordinate...
  RAGAS [12/100] What are IP options and why might they be used?...
  RAGAS [13/100] How does the book approach the teaching of jargon ...
  RAGAS [14/100] What role do netlink sockets play in Linux

INFO:openai._base_client:Retrying request to /chat/completions in 0.440544 seconds


  RAGAS [89/100] What is the primary purpose of the Linux Kernel Ne...
  RAGAS [90/100] How does the fixed point theorem play a role in th...
  RAGAS [91/100] Explain the process of IPv4 fragmentation and defr...
  RAGAS [92/100] What is the primary purpose of the master database...
  RAGAS [93/100] What are some of the practical applications of Mar...
  RAGAS [94/100] What is the significance of the "dotall" option in...
  RAGAS [95/100] How can you run a macro from the Visual Basic Edit...
  RAGAS [96/100] What is the book's stance on using triggers in SQL...
  RAGAS [97/100] What are the challenges in using naive Bayes model...
  RAGAS [98/100] What is the difference between call by name and ca...
  RAGAS [99/100] How does the book encourage the reader to engage w...
  RAGAS [100/100] How does the book introduce the concept of alterna...

  💰 Tokens RAGAS:
     Total  : 1,118,965  (prompt: 934,297 | completion: 184,668)
     TPM    : 53,888 tokens/min
     RPM    : 34 requests/min  

## 5. Coste total del experimento (indexación + evaluación)

In [10]:
import json

indexing_stats = indexing_tracker.to_dict()
query_stats    = result.token_usage

print("\n💰 COSTE TOTAL DEL EXPERIMENTO")
print("=" * 45)
print(f"\n  📥 Indexación:")
print(f"     LLM tokens  : {indexing_stats['llm']['total_tokens']:,}")
print(f"     Emb requests: {indexing_stats['embeddings']['requests']:,}")
print(f"     Tiempo       : {indexing_stats['elapsed_minutes']} min")

if query_stats:
    t = query_stats.get('total', {})
    r = query_stats.get('rates', {})
    print(f"\n  🔍 Evaluación RAGAS:")
    print(f"     Total tokens : {t.get('total_tokens', 0):,}")
    print(f"     TPM          : {r.get('tpm', 0):,}")
    print(f"     RPM          : {r.get('rpm', 0):,}")
print("=" * 45)

NameError: name 'indexing_tracker' is not defined

## 6. Inspeccionar respuestas individuales

In [10]:
for r in result.qa_results:
    print(f"❓ {r.question}")
    print(f"✅ GT:  {r.ground_truth[:150]}...")
    print(f"🤖 RAG: {r.answer[:150]}...")
    print(f"⏱️  {r.latency_s}s\n")

❓ What are the key features of this text that aid in learning object-oriented concepts in Java?
✅ GT:  Key features include an early introduction to object-oriented programming, the use of contour diagrams to illustrate object-oriented concepts, and the...
🤖 RAG: The text offers several features to help learners grasp object-oriented concepts in Java:

*   **Early Introduction to Objects**: The book introduces ...
⏱️  5.89s

❓ How does the text compare to other Java programming texts in terms of content and detail?
✅ GT:  The text aims to fill the gap between comprehensive texts that might cover too many details, making them difficult for beginners, and shortened introd...
🤖 RAG: The text aims to strike a balance between comprehensive coverage and conciseness, differing from other Java programming texts in its approach. Some te...
⏱️  7.5s

❓ What audience is the text primarily intended for?
✅ GT:  The text is intended primarily for readers who have not had any previous programming exp